In [134]:
import pandas as pd
import numpy as np

In [135]:
import kagglehub

# Download latest version
# path = kagglehub.dataset_download("lakshmi25npathi/imdb-dataset-of-50k-movie-reviews")

# print("Path to dataset files:", path)

In [136]:
import pandas as pd

path = r"C:\Users\saurabh upadhyay\.cache\kagglehub\datasets\lakshmi25npathi\imdb-dataset-of-50k-movie-reviews\versions\1\IMDB Dataset.csv"

df = pd.read_csv(path)

print(df.head())
print(df.shape)

                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive
(50000, 2)


In [137]:
def clean_data(data):
    # Remove any rows with missing values
    data = data.dropna()
    
    # Remove any duplicate rows
    data= data.drop_duplicates()
    
    # Convert text to lowercase
    data['review'] = data['review'].str.lower()
    
    return data




In [138]:
df=clean_data(df)

In [139]:
print(len(df['sentiment']))
print(len(df['review']))

49582
49582


In [140]:
from sklearn.preprocessing import LabelEncoder
lb=LabelEncoder()
df['sentiment']=lb.fit_transform(df['sentiment'])
df['sentiment'].head()

0    1
1    1
2    1
3    0
4    1
Name: sentiment, dtype: int64

In [141]:
# remove html ,patuncation

import re
import string
def remove_panct(data):
    for p in string.punctuation:
        data=data.replace(p," ")
    return data


In [142]:
df['review']=df['review'].apply(remove_panct)

df['review'][0]


'one of the other reviewers has mentioned that after watching just 1 oz episode you ll be hooked  they are right  as this is exactly what happened with me  br    br   the first thing that struck me about oz was its brutality and unflinching scenes of violence  which set in right from the word go  trust me  this is not a show for the faint hearted or timid  this show pulls no punches with regards to drugs  sex or violence  its is hardcore  in the classic use of the word  br    br   it is called oz as that is the nickname given to the oswald maximum security state penitentary  it focuses mainly on emerald city  an experimental section of the prison where all the cells have glass fronts and face inwards  so privacy is not high on the agenda  em city is home to many  aryans  muslims  gangstas  latinos  christians  italians  irish and more    so scuffles  death stares  dodgy dealings and shady agreements are never far away  br    br   i would say the main appeal of the show is due to the fa

In [143]:
# import nltk
# nltk.download('wordnet')
from nltk.stem import WordNetLemmatizer
lemmatizer=WordNetLemmatizer()
def lemmatizer_data(data):
    return "".join(lemmatizer.lemmatize(word)
    for word in data.split())


In [144]:
df['review']=df['review'].apply(lemmatizer_data)


In [145]:
len(df['review'][0])

1334

In [149]:
from sklearn.model_selection import train_test_split

x_train,x_temp,y_train,y_temp=train_test_split(df['review'],df['sentiment'],test_size=0.2,random_state=42,stratify=df['sentiment'])

x_val,x_test,y_val,y_test=train_test_split(x_temp,y_temp,test_size=0.5,random_state=42,stratify=y_temp)

In [150]:
# convert text into number +automaticly tokenizer

from sklearn.feature_extraction.text import TfidfVectorizer
tfidf=TfidfVectorizer(max_features=10000)
x_train=tfidf.fit_transform(x_train)
x_test=tfidf.transform(x_test)
x_val=tfidf.transform(x_val)

In [151]:
x_train=x_train. toarray()
x_val=x_val. toarray()
x_test=x_test. toarray()

In [152]:
import tensorflow as tf
from tensorflow import keras
from keras import layers
from keras.layers import Dropout,Dense
from keras.models import Sequential

model=Sequential()
model.add(Dense(128,activation='relu',input_shape=(x_train.shape[1],)))
model.add(Dropout(0.2))
model.add(Dense(64,activation='relu'))
model.add(Dropout(0.2))
model.add(Dense(32,activation='relu'))
model.add(Dropout(0.2))
model.add(Dense(2,activation='softmax'))

c:\Users\saurabh upadhyay\AppData\Local\Programs\Python\Python310\lib\site-packages\keras\src\layers\core\dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [153]:
model.summary()

Model: "sequential_11"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_42 (Dense)                │ (None, 128)            │     1,280,128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_31 (Dropout)            │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_43 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_32 (Dropout)            │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_44 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_33 (Dropout)            │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_45 (Dense)                │ (None, 2)              │            66 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,290,530 (4.92 MB)

 Trainable params: 1,290,530 (4.92 MB)

 Non-trainable params: 0 (0.00 B)

In [154]:
model.compile(optimizer='adam',loss='sparse_categorical_crossentropy',metrics=['accuracy'])
history=model.fit(x_train,y_train,epochs=10,batch_size=32,validation_data=(x_val,y_val))

MemoryError: Unable to allocate 1.48 GiB for an array with shape (39665, 10000) and data type float32

In [ ]:
model.evaluate(x_test,y_test)